In [38]:
import pandas as pd # main data tool (like Excel in Python)
import numpy as np # math & number operations
import matplotlib.pyplot as plt # basic charts
import seaborn as sns # nicer looking charts
import warnings
warnings.filterwarnings('ignore') # hide unnecessary warning messages
pd.set_option('display.max_columns', None) # show ALL columns
print('All libraries imported successfully!')

All libraries imported successfully!


In [39]:
df = pd.read_csv('D:\Technocolabs\Delivery Five Cities Datasets\delivery_yt.csv') # reads the file
print('File loaded!')
print('Number of rows :', df.shape[0]) # how many records
print('Number of columns :', df.shape[1]) # how many field

File loaded!
Number of rows : 206431
Number of columns : 17


In [40]:
df.isnull().sum()

order_id                0
region_id               0
city                    0
courier_id              0
lng                     0
lat                     0
aoi_id                  0
aoi_type                0
accept_time             0
accept_gps_time         0
accept_gps_lng       2422
accept_gps_lat       2422
delivery_time           0
delivery_gps_time       0
delivery_gps_lng        0
delivery_gps_lat        0
ds                      0
dtype: int64

In [41]:
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct.sort_values(ascending=False))

accept_gps_lat       1.173273
accept_gps_lng       1.173273
order_id             0.000000
accept_gps_time      0.000000
delivery_gps_lat     0.000000
delivery_gps_lng     0.000000
delivery_gps_time    0.000000
delivery_time        0.000000
accept_time          0.000000
region_id            0.000000
aoi_type             0.000000
aoi_id               0.000000
lat                  0.000000
lng                  0.000000
courier_id           0.000000
city                 0.000000
ds                   0.000000
dtype: float64


In [42]:
before = len(df)
df = df.dropna(subset=['accept_gps_lng', 'accept_gps_lat'])
df = df.reset_index(drop=True)
after = len(df)

print(f'Rows before : {before:,}')
print(f'Rows after  : {after:,}')
print(f'Rows removed: {before - after:,}')

Rows before : 206,431
Rows after  : 204,009
Rows removed: 2,422


In [43]:
df.isnull().sum().sum() == 0, 'Unexpected missing values found!'
print('Confirmed: No missing values in delivery_cq dataset.')

Confirmed: No missing values in delivery_cq dataset.


In [44]:
df['gps_synced_accept'] = df['accept_time'] == df['accept_gps_time']
df['gps_synced_delivery'] = df['delivery_time'] == df['delivery_gps_time']

In [45]:
print('GPS synced at accept :', df['gps_synced_accept'].sum())
print('GPS synced at delivery :', df['gps_synced_delivery'].sum())

GPS synced at accept : 204009
GPS synced at delivery : 204009


In [46]:
print(type(df['accept_time'][0])) # will show <class 'str'> = text
print(df['accept_time'][0]) # shows: 10-22 10:26:00

<class 'str'>
01/06/2018 08:00


In [47]:
# Step 1 — Convert datetime (different format from other datasets)
time_cols = ['accept_time', 'accept_gps_time', 'delivery_time', 'delivery_gps_time']

for col in time_cols:
    df[col] = pd.to_datetime(df[col], 
                             format='%d/%m/%Y %H:%M',  # ← new format
                             errors='coerce')

print(df[time_cols].dtypes)
# All should show datetime64[ns] ✅

# Step 2 — Verify conversion worked
print(df[time_cols].head(3))

accept_time          datetime64[ns]
accept_gps_time      datetime64[ns]
delivery_time        datetime64[ns]
delivery_gps_time    datetime64[ns]
dtype: object
          accept_time     accept_gps_time       delivery_time  \
0 2018-06-01 08:00:00 2018-06-01 08:00:00 2018-06-01 16:21:00   
1 2014-08-01 08:30:00 2014-08-01 08:30:00 2014-08-01 15:01:00   
2 2026-01-09 09:08:00 2026-01-09 09:08:00 2026-01-09 15:15:00   

    delivery_gps_time  
0 2018-06-01 16:21:00  
1 2014-08-01 15:01:00  
2 2026-01-09 15:15:00  


In [19]:
# Remove (only ~1.2% of 2,06,431 rows)
df = df.dropna(subset=['accept_gps_lng', 'accept_gps_lat']).reset_index(drop=True)
print('Rows after removing missing GPS:', len(df))
# ~204,009

Rows after removing missing GPS: 204009


In [48]:
print('After conversion:')
print(df[time_cols].dtypes)

After conversion:
accept_time          datetime64[ns]
accept_gps_time      datetime64[ns]
delivery_time        datetime64[ns]
delivery_gps_time    datetime64[ns]
dtype: object


In [50]:
# Example: How many minutes from accept to delivery?
df['accept_to_delivery_min'] = (
(df['delivery_time'] - df['accept_time']).dt.total_seconds() / 60)

In [51]:
print(df['accept_to_delivery_min'].describe().round(1))

count      204009.0
mean          662.9
std       1706685.9
min     -52070929.0
25%           113.0
50%           206.0
75%           342.0
max      50371199.0
Name: accept_to_delivery_min, dtype: float64


In [52]:
duplicates = df.duplicated(subset=['order_id'])
print('Number of duplicate rows:', duplicates.sum())

Number of duplicate rows: 0


In [53]:
before = len(df)
df = df.drop_duplicates(subset=['order_id'], keep='first')
df = df.reset_index(drop=True) # re-number the rows from 0
after = len(df)
print(f'Rows before: {before:,}')
print(f'Rows after : {after:,}')
print(f'Duplicates removed: {before - after:,}')

Rows before: 204,009
Rows after : 204,009
Duplicates removed: 0


In [27]:
col = 'accept_to_delivery_min'
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = (df[col] < lower) | (df[col] > upper)
print(f'Outlier rows found: {outliers.sum():,}')
print(f'Valid range: {lower:.0f} to {upper:.0f} minutes')

Outlier rows found: 5,676
Valid range: -230 to 686 minutes


In [54]:
df = df[df['accept_to_delivery_min'] >= 0].reset_index(drop=True)
print('Rows after removing negatives:', len(df))

Rows after removing negatives: 203792


In [55]:
df['accept_to_delivery_min'] = df['accept_to_delivery_min'].clip(upper=upper)
print('Max duration after clipping:', df['accept_to_delivery_min'].max())


Max duration after clipping: 685.5


In [56]:
# Cell 10a — Extract hour, day, month from accept_time
df['accept_hour'] = df['accept_time'].dt.hour # 0 to 23
df['accept_day'] = df['accept_time'].dt.day # 1 to 31
df['accept_month'] = df['accept_time'].dt.month # 1 to 12
print(df[['accept_time','accept_hour','accept_day','accept_month']].head(3))
# Cell 10b — Group hour into time-of-day buckets
def time_of_day(hour):
    if 0 <= hour < 6: return 'Night'
    elif 6 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else: return 'Night'
df['time_of_day'] = df['accept_hour'].apply(time_of_day)
print(df['time_of_day'].value_counts())

          accept_time  accept_hour  accept_day  accept_month
0 2018-06-01 08:00:00            8           1             6
1 2014-08-01 08:30:00            8           1             8
2 2026-01-09 09:08:00            9           9             1
time_of_day
Morning      169324
Afternoon     31435
Evening        2766
Night           267
Name: count, dtype: int64


In [57]:
df['gps_drift'] = ((
(df['delivery_gps_lng'] - df['accept_gps_lng'])**2 +
(df['delivery_gps_lat'] - df['accept_gps_lat'])**2
) ** 0.5)
print(df['gps_drift'].describe().round(4))

count    203792.0000
mean          0.1275
std           2.9172
min           0.0000
25%           0.0239
50%           0.0432
75%           0.0802
max         126.4487
Name: gps_drift, dtype: float64


In [58]:
df['delivery_speed_proxy'] = df['gps_drift'] / (df['accept_to_delivery_min'] + 1)
print('Speed proxy stats:')
print(df['delivery_speed_proxy'].describe().round(4))

Speed proxy stats:
count    203792.0000
mean          0.0012
std           0.0462
min           0.0000
25%           0.0001
50%           0.0002
75%           0.0004
max           7.4242
Name: delivery_speed_proxy, dtype: float64


In [59]:
df['delivery_month'] = df['delivery_time'].dt.month
print(df['delivery_month'].value_counts().sort_index())

delivery_month
1      5587
2      6767
3      7305
4      7077
5     25343
6     28569
7     26396
8     28585
9     25243
10    29948
11     6457
12     6515
Name: count, dtype: int64


In [60]:
print('Final dataset shape:', df.shape)
print()
print('Missing values left:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print('New columns created:')
new_cols = ['gps_synced_accept', 'gps_synced_delivery',
'accept_to_delivery_min', 'accept_hour', 'accept_day',
'accept_month', 'time_of_day', 'gps_drift',
'delivery_speed_proxy', 'delivery_month']
print(new_cols)

Final dataset shape: (203792, 27)

Missing values left:
Series([], dtype: int64)

New columns created:
['gps_synced_accept', 'gps_synced_delivery', 'accept_to_delivery_min', 'accept_hour', 'accept_day', 'accept_month', 'time_of_day', 'gps_drift', 'delivery_speed_proxy', 'delivery_month']


In [35]:
df.head()

,order_id,region_id,city,courier_id,lng,lat,aoi_id,aoi_type,accept_time,accept_gps_time,accept_gps_lng,accept_gps_lat,delivery_time,delivery_gps_time,delivery_gps_lng,delivery_gps_lat,ds,accept_to_delivery_min,accept_hour,accept_day,accept_month,time_of_day,gps_drift,delivery_speed_proxy,delivery_month
0,2071340,86,Yantai,3135,121.40538,37.35796,175,14,2018-06-01 08:00:00,2018-06-01 08:00:00,121.38013,37.41137,2018-06-01 16:21:00,2018-06-01 16:21:00,121.35896,37.37456,618,501.0,8,1,6,Morning,0.042463,0.000085,6
1,3767291,86,Yantai,3135,121.40539,37.35804,175,14,2014-08-01 08:30:00,2014-08-01 08:30:00,121.38042,37.41050,2014-08-01 15:01:00,2014-08-01 15:01:00,121.40707,37.35897,814,391.0,8,1,8,Morning,0.058013,0.000148,8
2,2184812,86,Yantai,3135,121.40533,37.35809,175,14,2026-01-09 09:08:00,2026-01-09 09:08:00,121.39184,37.41000,2026-01-09 15:15:00,2026-01-09 15:15:00,121.40824,37.35837,901,367.0,9,9,1,Morning,0.054172,0.000147,1
3,4048066,86,Yantai,1586,121.51762,37.45354,293,1,2025-08-01 13:59:00,2025-08-01 13:59:00,121.38107,37.41032,2025-08-01 18:26:00,2025-08-01 18:26:00,121.52112,37.45253,825,267.0,13,1,8,Afternoon,0.146273,0.000546,8
4,2309133,86,Yantai,1508,121.40041,37.41166,314,14,2026-09-07 08:57:00,2026-09-07 08:57:00,121.38042,37.41083,2026-09-07 10:56:00,2026-09-07 10:56:00,121.40067,37.41131,709,119.0,8,7,9,Morning,0.020256,0.000169,9


In [61]:
df.to_csv('delivery_yt_cleaned.csv', index=False)
print('File saved as: delivery_yt_cleaned.csv')
print(f'Total rows saved: {len(df):,}')

File saved as: delivery_yt_cleaned.csv
Total rows saved: 203,792
